###### Content under Creative Commons Attribution license CC-BY 4.0, code under BSD 3-Clause License © 2022  by D. Koehn, T. Meier and J. Stampa, notebook style sheet by L.A. Barba, N.C. Clementi

# Digital Signal Processing in Geophysics 

## Chapter 4: Data Pre-Processing

### 4.4 Simple Transformations

Simple transformations are often helpful in making the desired signal more visible or in eliminating errors. If you are primarily interested in the high frequencies and the low frequencies are more of a nuisance, you can apply a time-domain differentiation to the data series—several times if necessary. Conversely, integration with respect to time is helpful when the low frequencies are of greater interest. However, integration can also amplify long-period noise, which is why combining it with filtering is recommended. Another example is amplitude correction, e.g., in reflection seismic measurements: the amplitude decay over time is corrected to make weak later arrivals more visible and to artificially induce stationarity in the time series with respect to amplitude.

If the amplitude is determined only up to an unknown factor, e.g., due to unknown coupling of a geophone to the subsurface, the time series is normalized to the maximum of the data series. The waveform is then interpreted, but not its absolute value. This is sufficient for many applications, since often only travel times or the phase of the signal - but not the amplitude - are evaluated.
 

To make smaller signals visible, the data series can be logarithmized:

\begin{equation}
\tilde{x}_i = \ln {x_i}. \tag{4.4}
\end{equation}

Let’s take a closer look at this with an example. As usual, we’ll start by importing some Python libraries.

In [ ]:
# Importiere Python Bibliotheken 
# ------------------------------
#%matplotlib inline
from ipywidgets import interactive
import matplotlib.pyplot as plt
import numpy as np

In the next step, we will use the sine function from Exercise 3.4 and add several high-amplitude spikes, whose amplitudes we can adjust interactively. A toggle switch allows you to enable logarithmic transformation of the time series.

In [ ]:
def x_log(spike_amp,logx):
    
    # Define parameters
    T = 25.                  # period 1 [s]
    dt = .1                  # time sampling [s]
    L = 1000.                # length of the time series [s]

    # Define sine function ...
    omega = 2. * np.pi / T   # compute circular frequency from period
    t = np.arange(0,L+dt,dt) # compute time vector
    x = np.sin(omega*t)  # compute sine wave
    
    # Add spike to the time series
    N = len(x)
    x[N//2] = spike_amp
    
    if(logx==True):
        
        # if x > 0
        x = np.log(x)
    
        # if x <=0
        #C = 1 / np.log(10)
        #x = np.sign(x) * np.log(1+np.abs(x/C))
    
    # Initialize Plots
    plt.figure(figsize=(10,5))
    
    # Plot time series
    plt.plot(t, x, 'b')
    plt.xlabel('Time (s)')
    plt.ylabel('f(t) (s)')
    
    if(logx==True):
        plt.ylabel('log(f(t)) (s)')
        
    plt.title('Time Series f(t) = Sin(t) + Spike(t)')

Next, we can play around with the time series we've generated and the logarithmic transformation...

In [ ]:
interactive_plot = interactive(x_log, spike_amp=(0., 1000.,50), logx=False)
output = interactive_plot.children[-1]
output.layout.height = '500px'
interactive_plot

However, this is a nonlinear transformation. Any linear relationships that may exist are thereby destroyed. Further digital processing of the data is then usually no longer appropriate. Another nonlinear transformation that can be used very flexibly to either make weak signals visible or suppress interfering background noise is:

\begin{equation}
\tilde{x}_i = sgn(x_i) |x_i|^{n}. \tag{4.5}
\end{equation} 

For $n = 1$, there is no change; for $n<1$, however, smaller amplitudes are amplified. Due to its flexibility, this transformation is particularly suitable for representing time series with a wide dynamic range. For $n>1$, weaker signals are suppressed.

Let’s test transformation (4.5) on the time series with the sinusoidal oscillation and the spike.

In [ ]:
def sgnxxn(n,sgnxxn):
    
    # Define parameters
    T = 25.                  # period 1 [s]
    dt = .1                  # time sampling [s]
    L = 1000.                # length of the time series [s]
    spike_amp = 1000.        # spike amplitude

    # Define sine function ...
    omega = 2. * np.pi / T   # compute circular frequency from period
    t = np.arange(0,L+dt,dt) # compute time vector
    x = np.sin(omega*t)  # compute sine wave
    
    # Add spike to the time series
    N = len(x)
    x[N//2] = spike_amp
    
    if(sgnxxn==True):
        x = np.sign(x) * np.abs(x)**n
    
    # Initialize Plots
    plt.figure(figsize=(10,5))
    
    # Plot time series
    plt.plot(t, x, 'b')
    plt.xlabel('Time (s)')
    plt.ylabel('f(t) (s)')
    
    if(sgnxxn==True):
        plt.ylabel('sgn(f(t))abs(f(t))**n (s)')
        
    plt.title('Time Series f(t) = Sin(t) + Spike(t)')

In [ ]:
interactive_plot = interactive(sgnxxn, n=(0., 2., .1), sgnxxn=False)
output = interactive_plot.children[-1]
output.layout.height = '500px'
interactive_plot

Sometimes, the sequence of values is also reduced to a binary signal in order to ensure stationarity and eliminate large differences in amplitude:

\begin{equation}
\tilde{x}_i = \begin{cases} 1,& x_i \geq 0 \\ -1,& x_i < 0. \end{cases} \tag{4.6}
\end{equation}

Surprisingly, this highly nonlinear transformation for establishing [stationarity](https://en.wikipedia.org/wiki/Stationary_process) can be successfully used to determine the Green’s function from the background noise.

To test whether a time series is stationary, one can use the [Dickey-Fuller test](https://pythondata.com/stationary-data-tests-for-time-series-forecasting/). To do this, we simply need to import the appropriate Python function ...

In [ ]:
from statsmodels.tsa.stattools import adfuller

To test whether the transformation $(4.6)$ produces stationarity, we add a linear trend with slope $b$ to our sine function with amplitude $a$

In [ ]:
def sgnx(a,b,sgnx):
    
    # Define parameters
    T = 25.                  # period 1 [s]
    dt = .1                  # time sampling [s]
    L = 1000.                # length of the time series [s]

    # Define sine function ...
    omega = 2. * np.pi / T   # compute circular frequency from period
    t = np.arange(0,L+dt,dt) # compute time vector
    x = a*np.sin(omega*t)  # compute sine wave
    
    # Add linear trend to the time series
    xlin = b * t
    x = x + xlin
        
    if(sgnx==True):
        x = np.sign(x)
    
    # Apply Dickey-Fuller test to x
    res = adfuller(x)
    
    # Initialize Plots
    plt.figure(figsize=(10,5))
    
    # Plot time series
    plt.plot(t, x, 'b')
    plt.xlabel('Time (s)')
    plt.ylabel('f(t) (s)')
    
    if(sgnx==True):
        plt.ylabel('sgn(f(t)) (s)')
    
    title = 'Time Series f(t) = a*Sin(t) + b*t, Dickey Fuller Test p-value = ' + str(res[1])
    
    plt.title(title)

Strictly speaking, in the `sgnx` function, we did not implement Eq. (4.6) but rather the `NumPy` function [sign](https://numpy.org/doc/stable/reference/generated/numpy.sign.html).

In [ ]:
interactive_plot = interactive(sgnx, a=(0., 10., .1), b=(-.05, .05, .005),  sgnx=False)
output = interactive_plot.children[-1]
output.layout.height = '500px'
interactive_plot

For the sine function without a linear trend ($b=0$), the Dickey-Fuller test yields a p-value of 0, indicating that we have a stationary time series. However, even with small variations of $b \ne 0$, the p-values are close to 1, meaning the time series is no longer stationary.
 

By applying the transformation $sgn(x)$, the p-value immediately jumps back to zero, demonstrating that the sign transformation can establish stationarity. In this case, however, a significant amount of information is lost. A better approach is to apply a trend correction, which is discussed in more detail in Section 4.5.

The following transformation can be applied to detect a signal:

\begin{equation}
\tilde{x}_i = \begin{cases} 0,& x_i < \mbox{threshold xc} \\ 1,& x_i \geq \mbox{threshold xc}. \end{cases} \tag{4.7}
\end{equation}

The result is equal to one if the amplitude of the data series exceeds a threshold, thereby indicating the presence of a signal.

Let’s test this again for our sine time series, isolating a portion at the beginning of the time series, attenuating the amplitudes with a Gaussian function, and adding some randomly distributed noise.

In [ ]:
def sgndetect(xc,an,detect,damp,noise):
    
    # Define parameters
    T = 25.                  # period 1 [s]
    dt = .1                  # time sampling [s]
    L = 1000.                # length of the time series [s]

    # Define sine function ...
    omega = 2. * np.pi / T   # compute circular frequency from period
    t = np.arange(0,L+dt,dt) # compute time vector
    x = np.sin(omega*t)  # compute sine wave
    
    # Isolate a part of the time series 
    N = len(x)
    t1 = (int) (50 // dt)
    t2 = (int) (100 // dt)
    t3 = (int) (200 // dt)
    x[:t1] = 0.
    x[t2:t3] = 0.
    
    # Damp time series with Gaussian function before t3
    if(damp==True):
        a=1e-6
        xdamp = x * np.exp(-a*(t-t3)**2)
        x[:t3] = xdamp[:t3]
        
    # Add normal distributed noise to the time series
    if(noise==True):
        mu = 0.
        sigma = .1
        xnoise = np.random.normal(mu, sigma, N)
        x = x + an * xnoise
    
    if(detect==True):
        x[x>=xc] = 1.
        x[x<xc] = 0.
    
    # Initialize Plots
    plt.figure(figsize=(10,5))
    
    # Plot time series
    plt.plot(t, x, 'b')
    plt.xlabel('Time (s)')
    plt.ylabel('f(t) (s)')
    
    if(detect==True):
        plt.ylabel('sgn(f(t)) (s)')
    
    title = 'Time Series f(t) = Sin(t)'
    
    plt.title(title)

In [ ]:
interactive_plot = interactive(sgndetect, xc=(0., .04, .005), an=(.0, .2, .01), detect=False, damp=False, noise=False)
output = interactive_plot.children[-1]
output.layout.height = '500px'
interactive_plot

Sometimes, multiplying the data series by a constant factor is sufficient to correct errors. The erroneous data series need only be multiplied by a correction factor $a$, which, however, is unknown. To determine it, a model for the data series is assumed. The corrected data series $\hat{x}_i$ is then given by:

\begin{equation}
\hat{x}_i = a \tilde{x}_i, \tag{4.8}
\end{equation} 

where $\tilde{x}_i$ is the uncorrected, measured data series.

We want to examine this using a time series similar to the Huddle test. Let us assume that $x_i$ denotes the seismogram of a calibrated reference seismometer. $\tilde{x}_i$, on the other hand, is the uncorrected time series of an uncalibrated seismometer. The two time series differ only by a correction factor $a$.
 
After applying the correction in Eq. (4.8) with the correct factor $a$ to the seismogram of the uncalibrated seismometer, we obtain the seismogram of the reference seismometer $x_i$.

The time series recorded with the Huddle Test for the reference seismometer $x_i$ and the uncalibrated seismometer $\tilde{x}_i$ consist of a superposition of two sine functions with periods of $T_1 = 25 s$ and $T_2 = 500 s$.

In [ ]:
def fscale(a, diff):
    
    # Define parameters
    T1 = 25.                 # period 1 [s]
    T2 = 500.                 # period 2 [s]
    dt = .1                  # time sampling [s]
    L = 1000.                # length of the time series [s]

    # Define sine functions ...
    omega1 = 2. * np.pi / T1   # compute circular frequency from period 1
    omega2 = 2. * np.pi / T2   # compute circular frequency from period 2
    t = np.arange(0,L+dt,dt) # compute time vector
    
    x1 = np.sin(omega1*t)  # compute sine wave 1
    x2 = np.sin(omega2*t)  # compute sine wave 1
    
    # Add both sine functions
    x = x1 + x2
    
    # Define corrected time series
    xb = x * a
    
    # Define reference time series
    ar = 1.27
    xh = x * ar
    
    # Initialize Plots
    plt.figure(figsize=(10,5))
    
    # Plot time series
    if(diff==False):
        plt.plot(t, xb, 'b', label = r'corrected time series $\hat{x}$')
        plt.plot(t, xh, 'r', label = r'reference time series $x$')
        plt.xlabel('Time t (s)')
        plt.ylabel('$x(t)$, $\hat{x}(t)$ (s)')
        plt.legend()
        title = 'Reference seismogram $x$ vs. corrected seismogram $\hat{x}$'

    if(diff==True):
        plt.plot(t, xh-xb, 'r', label = r'difference time series $x$ - $\hat{x}$')
        plt.xlabel('Time t (s)')
        plt.ylabel('$x(t)$ - $\hat{x}(t)$ (s)')
        plt.legend()
        
        # Compute least squares objective function value
        E = np.sum((xh-xb)**2)
        title = 'Difference $x$ - $\hat{x}$ (E = ' + np.array2string(E) + ')'         
    
    plt.title(title)

Durch Variation des Faktors $a$ können wir die unkorrigierte Zeitreihe $\overline{x}$ in die korrigierte Zeitreihe $\hat{x}$ transformieren. 

In [ ]:
interactive_plot = interactive(fscale, a=(.0, 10., .1), diff=False)
output = interactive_plot.children[-1]
output.layout.height = '500px'
interactive_plot

Assuming that $x_i$ has been measured by a nearby reference seismometer and can therefore be considered known, the correction factor $a$ can be determined using the method of least squares. In this method, the sum of squared errors

\begin{equation}
E = \sum_{i=1}^N (x_i - \hat{x}_i)^2 \tag{4.9}
\end{equation} 

is minimized. That is, the following should hold for the correction factor:

\begin{equation}
\frac{\partial E}{\partial a}\stackrel{!}{=}0. \tag{4.10}
\end{equation}

This condition leads to a simple determining equation for $a$. Multiplying $\tilde{x}$ by $a$ yields the corrected data series. 

A similar approach can be taken if a disturbance variable is present in a sequence of values $x_i$ that can be identified and measured by an external measurement $p_i$. In the simplest case, a linear model, specifically 

\begin{equation}
\hat{x}_i=f(p_i) = ap_i. \tag{4.11}
\end{equation}

Here, $x_i$ can be, for example, a continuous gravity measurement and $p_i$ the measurement of air pressure. $\hat{x}_i$ is the portion of the gravity measurement caused by the air pressure. If the factor $a$ is known, which describes the extent to which the disturbance variable appears in the measured data series, the external measurement of the disturbance variable $p_i$ can be multiplied by $a$ and subtracted from the measured data series to obtain the corrected data series. Often this is not the case, and it must be determined using the method of least squares. The sum of squared errors in this case is: 

\begin{equation}
E = \sum_{i=1}^N (x_i - a p_i)^2. \tag{4.12}
\end{equation} 

Minimizing this leads to a determination equation for $a$ similar to that in the previous example.

## Summary:

Using simple transformations, it is possible to ... 

- make weak signals visible

- establish the stationarity of time series

- detect signals 

- correct for amplitude differences